In [1]:
from pathlib import Path
import pynucastro as pyna
import pandas as pd
import random
import matplotlib.pyplot as plt
import numpy as np
%matplotlib widget

In [2]:
rates_reaclib_18_9_20=pyna.rates.library.Library(libfile=r"Nuclear_data\decays\actual\Reaclib_18_9_20")
rates_reaclib_18_9_20_Experimental=pyna.rates.library.Library(libfile=r"Nuclear_data\decays\actual\Reaclib_default_Experimental")
rates_alpha_no_T=pyna.rates.library.Library(libfile=r"Nuclear_data\decays\example_reaclib")
rates_beta_empirical=pyna.rates.library.Library(libfile=r"Nuclear_data\decays\empi_example")


In [3]:
sources_label_default=[]
sources_default=[]
number_default={}
for r in rates_reaclib_18_9_20.get_rates():
    if r.source['Label'] not in sources_label_default:
        sources_label_default.append(r.source['Label'])
        sources_default.append(r.source)
        number_default[r.source['Label']]=1
    else:
        number_default[r.source['Label']]+=1

sources_label_experimental=[]
sources_experimental=[]
number_experimental={}
for r in rates_reaclib_18_9_20_Experimental.get_rates():
    if r.source['Label'] not in sources_label_experimental:
        sources_label_experimental.append(r.source['Label'])
        sources_experimental.append(r.source)
        number_experimental[r.source['Label']]=1
    else:
        number_experimental[r.source['Label']]+=1

sources_label_theoretical=[]
sources_theoretical=[]
number_theoretical={}
for rate in rates_reaclib_18_9_20.get_rates():
    if rate.source['Label'] not in sources_label_experimental:
        if rate.source['Label']  not in sources_label_theoretical:
            sources_label_theoretical.append(rate.source['Label'] )
            sources_theoretical.append(rate.source)
            number_theoretical[rate.source['Label']]=1
        else:
            number_theoretical[rate.source['Label']]+=1


In [4]:
#encuentro las teoricas beta
filter_beta=pyna.RateFilter(max_reactants=1,
                            max_products=1,
                            filter_function=lambda r: r.source['Label'] in sources_label_theoretical and r.Q>0 and r.reactants[0].Z+1==r.products[0].Z and r.reactants[0].A==r.products[0].A
                            )

#encuentro las teóricas alpha
filter_alpha=pyna.RateFilter(products=['he4'],
                             exact=False,
                             max_reactants=1,
                             max_products=2,
                             filter_function=lambda r: r.source['Label'] in sources_label_theoretical and r.Q>0 and r.reactants[0].Z==r.products[1].Z+r.products[0].Z and r.reactants[0].A==r.products[1].A+r.products[0].A
                             )
 
rates_reaclib_18_9_20_alpha=rates_reaclib_18_9_20.filter(filter_alpha)
rates_reaclib_18_9_20_beta=rates_reaclib_18_9_20.filter(filter_beta)


In [5]:
print(len(rates_reaclib_18_9_20_alpha.get_rates()),len(rates_alpha_no_T.get_rates()))

1121 1121


In [6]:
for i in range(len(rates_reaclib_18_9_20_alpha.get_rates())):
    rates_reaclib_18_9_20.remove_rate(rates_reaclib_18_9_20_alpha.get_rates()[i])
    rates_reaclib_18_9_20.add_rate(rates_alpha_no_T.get_rates()[i])



In [ ]:

def reaclib2_to_reaclib1(input_file, output_file):
    """
    Convert a Reaclib v2 format file into Reaclib v1 format.
    
    Parameters
    ----------
    input_file : str or Path
        Path to Reaclib v2 file
    output_file : str or Path
        Path to write Reaclib v1 file
    """
    input_file = Path(input_file)
    output_file = Path(output_file)

    with input_file.open("r") as fin, output_file.open("w") as fout:
        header=[]
        head=0
        for line in fin:
            if len(line) != 75:
                if line[0] != head:
                    if line[0] not in header:
                        head=line[0]
                        header.append(line[0])
                        fout.write(line[0]+' '*73 + '\n')
                        fout.write(' '*74 + '\n')
                        fout.write(' '*74 + '\n')
                    else:
                else:
                    continue
            else:
                fout.write(line)
   

def reaclib2_to_reaclib1_T1GK(input_file, output_file):
    """
    Convert a Reaclib v2 format file into Reaclib v1 format.
    
    Parameters
    ----------
    input_file : str or Path
        Path to Reaclib v2 file
    output_file : str or Path
        Path to write Reaclib v1 file
    """
    input_file = Path(input_file)
    output_file = Path(output_file)

    with input_file.open("r") as fin, output_file.open("w") as fout:
        header=[]
        head=0
        for line in fin:
            
            if len(line) != 75:
                if line[0] != head:
                    head=line[0]
                    header.append(line[0])
                    fout.write(line[0]+' '*73 + '\n')
                    fout.write(' '*74 + '\n')
                    fout.write(' '*74 + '\n')
                else:
                    continue
            else:
                fout.write(line)

file=Path(r"Nuclear_data\decays\simulations_several\reaclib_alpha_no_T")
open(file,'w')
rates_reaclib_18_9_20.write_to_file(file)
reaclib2_to_reaclib1(r"Nuclear_data\decays\simulations_several\reaclib_alpha_no_T",
                     r"Nuclear_data\decays\simulations_several\reaclib_alpha_no_T_reaclib1")
      